<a href="https://colab.research.google.com/github/jennyk23/Magpy/blob/main/atividade4_sistema_academico_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade 4 — Sistema Acadêmico
Execute a célula abaixo. Ela instala o MariaDB, cria o banco, insere os dados, atualiza os status e exclui matrículas canceladas.

In [1]:
# ATIVIDADE 4 – EXECUÇÃO COMPLETA NO GOOGLE COLAB

import os
import shutil
import subprocess
import time
from pathlib import Path

# 1. Instala o MariaDB se necessário
if shutil.which("mariadb") is None:
    print("Instalando MariaDB...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(
        ["apt-get", "install", "-y", "mariadb-server", "mariadb-client"],
        check=True,
        stdout=subprocess.DEVNULL
    )

# 2. Inicia o servidor
os.makedirs("/run/mysqld", exist_ok=True)
subprocess.run(["chown", "mysql:mysql", "/run/mysqld"], check=False)
subprocess.run(
    ["service", "mariadb", "start"],
    check=False,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

for _ in range(20):
    teste = subprocess.run(
        ["mariadb-admin", "-u", "root", "ping", "--silent"],
        text=True,
        capture_output=True
    )
    if teste.returncode == 0:
        break
    time.sleep(1)
else:
    raise RuntimeError("Não foi possível iniciar o MariaDB.")

print("MariaDB iniciado com sucesso.")

# 3. Cria o arquivo SQL
sql = "-- ============================================================\n-- ATIVIDADE 4 – SISTEMA ACADÊMICO\n-- MYSQL / MARIADB\n-- ============================================================\n\nDROP DATABASE IF EXISTS sistema_academico;\n\nCREATE DATABASE sistema_academico\n    CHARACTER SET utf8mb4\n    COLLATE utf8mb4_unicode_ci;\n\nUSE sistema_academico;\n\n-- ============================================================\n-- 1. TABELA ALUNO\n-- ============================================================\n\nCREATE TABLE aluno (\n    id_aluno INT AUTO_INCREMENT,\n    nome VARCHAR(150) NOT NULL,\n\n    CONSTRAINT pk_aluno\n        PRIMARY KEY (id_aluno),\n\n    CONSTRAINT chk_aluno_nome\n        CHECK (CHAR_LENGTH(TRIM(nome)) > 0)\n) ENGINE = InnoDB;\n\n-- ============================================================\n-- 2. TABELA CURSO\n-- ============================================================\n\nCREATE TABLE curso (\n    id_curso INT AUTO_INCREMENT,\n    nome VARCHAR(150) NOT NULL,\n\n    CONSTRAINT pk_curso\n        PRIMARY KEY (id_curso),\n\n    CONSTRAINT uq_curso_nome\n        UNIQUE (nome),\n\n    CONSTRAINT chk_curso_nome\n        CHECK (CHAR_LENGTH(TRIM(nome)) > 0)\n) ENGINE = InnoDB;\n\n-- ============================================================\n-- 3. TABELA MATRÍCULA\n-- Um aluno pode estar matriculado em vários cursos\n-- ============================================================\n\nCREATE TABLE matricula (\n    id_aluno INT NOT NULL,\n    id_curso INT NOT NULL,\n    status VARCHAR(20) NOT NULL DEFAULT 'Cursando',\n\n    CONSTRAINT pk_matricula\n        PRIMARY KEY (id_aluno, id_curso),\n\n    CONSTRAINT fk_matricula_aluno\n        FOREIGN KEY (id_aluno)\n        REFERENCES aluno(id_aluno)\n        ON UPDATE CASCADE\n        ON DELETE CASCADE,\n\n    CONSTRAINT fk_matricula_curso\n        FOREIGN KEY (id_curso)\n        REFERENCES curso(id_curso)\n        ON UPDATE CASCADE\n        ON DELETE RESTRICT,\n\n    CONSTRAINT chk_matricula_status\n        CHECK (\n            status IN (\n                'Cursando',\n                'Aprovado',\n                'Reprovado',\n                'Cancelado'\n            )\n        )\n) ENGINE = InnoDB;\n\n-- ============================================================\n-- 4. TABELA NOTA\n-- ============================================================\n\nCREATE TABLE nota (\n    id_nota INT AUTO_INCREMENT,\n    id_aluno INT NOT NULL,\n    valor DECIMAL(4,2) NOT NULL,\n\n    CONSTRAINT pk_nota\n        PRIMARY KEY (id_nota),\n\n    CONSTRAINT fk_nota_aluno\n        FOREIGN KEY (id_aluno)\n        REFERENCES aluno(id_aluno)\n        ON UPDATE CASCADE\n        ON DELETE CASCADE,\n\n    CONSTRAINT chk_nota_valor\n        CHECK (valor >= 0 AND valor <= 10)\n) ENGINE = InnoDB;\n\n-- ============================================================\n-- 5. ALTERAÇÃO DA TABELA MATRÍCULA\n-- ============================================================\n\nALTER TABLE matricula\nADD COLUMN data_matricula DATE;\n\n-- ============================================================\n-- 6. INSERÇÃO DOS ALUNOS\n-- ============================================================\n\nINSERT INTO aluno (nome) VALUES\n('Ana Beatriz Souza'),\n('Bruno Henrique Lima'),\n('Carla Mendes Rocha');\n\n-- ============================================================\n-- 7. INSERÇÃO DOS CURSOS\n-- ============================================================\n\nINSERT INTO curso (nome) VALUES\n('Tecnologia da Informação'),\n('Inteligência Artificial'),\n('Engenharia de Controle e Automação');\n\n-- ============================================================\n-- 8. INSERÇÃO DAS MATRÍCULAS\n-- ============================================================\n\nINSERT INTO matricula (\n    id_aluno,\n    id_curso,\n    status,\n    data_matricula\n) VALUES\n(1, 1, 'Cursando', '2026-02-10'),\n(1, 2, 'Cursando', '2026-02-10'),\n(2, 1, 'Cursando', '2026-02-11'),\n(3, 3, 'Cancelado', '2026-02-12');\n\n-- ============================================================\n-- 9. INSERÇÃO DAS NOTAS\n-- ============================================================\n\nINSERT INTO nota (id_aluno, valor) VALUES\n(1, 8.00),\n(1, 9.00),\n(2, 5.00),\n(2, 6.00),\n(3, 7.00),\n(3, 8.00);\n\n-- ============================================================\n-- 10. CONSULTA DAS MÉDIAS\n-- ============================================================\n\nSELECT\n    a.id_aluno,\n    a.nome,\n    ROUND(AVG(n.valor), 2) AS media\nFROM aluno AS a\nINNER JOIN nota AS n\n    ON n.id_aluno = a.id_aluno\nGROUP BY\n    a.id_aluno,\n    a.nome\nORDER BY a.id_aluno;\n\n-- ============================================================\n-- 11. ATUALIZAÇÃO DOS STATUS\n-- Média >= 7: Aprovado\n-- Média < 7: Reprovado\n-- Cancelado não é alterado\n-- ============================================================\n\nUPDATE matricula AS m\nINNER JOIN (\n    SELECT\n        id_aluno,\n        AVG(valor) AS media\n    FROM nota\n    GROUP BY id_aluno\n) AS medias\n    ON medias.id_aluno = m.id_aluno\nSET m.status = CASE\n    WHEN medias.media >= 7 THEN 'Aprovado'\n    ELSE 'Reprovado'\nEND\nWHERE m.status <> 'Cancelado';\n\n-- ============================================================\n-- 12. CONFERÊNCIA DOS STATUS\n-- ============================================================\n\nSELECT\n    a.nome AS aluno,\n    c.nome AS curso,\n    m.data_matricula,\n    m.status,\n    ROUND(AVG(n.valor), 2) AS media\nFROM matricula AS m\nINNER JOIN aluno AS a\n    ON a.id_aluno = m.id_aluno\nINNER JOIN curso AS c\n    ON c.id_curso = m.id_curso\nLEFT JOIN nota AS n\n    ON n.id_aluno = a.id_aluno\nGROUP BY\n    a.id_aluno,\n    a.nome,\n    c.id_curso,\n    c.nome,\n    m.data_matricula,\n    m.status\nORDER BY\n    a.nome,\n    c.nome;\n\n-- ============================================================\n-- 13. EXCLUSÃO DAS MATRÍCULAS CANCELADAS\n-- ============================================================\n\nDELETE FROM matricula\nWHERE status = 'Cancelado';\n\n-- ============================================================\n-- 14. CONFERÊNCIA FINAL\n-- ============================================================\n\nSELECT\n    a.nome AS aluno,\n    c.nome AS curso,\n    m.data_matricula,\n    m.status\nFROM matricula AS m\nINNER JOIN aluno AS a\n    ON a.id_aluno = m.id_aluno\nINNER JOIN curso AS c\n    ON c.id_curso = m.id_curso\nORDER BY\n    a.nome,\n    c.nome;\n\nSELECT\n    'Alunos' AS tabela,\n    COUNT(*) AS quantidade\nFROM aluno\n\nUNION ALL\n\nSELECT\n    'Cursos',\n    COUNT(*)\nFROM curso\n\nUNION ALL\n\nSELECT\n    'Matrículas',\n    COUNT(*)\nFROM matricula\n\nUNION ALL\n\nSELECT\n    'Notas',\n    COUNT(*)\nFROM nota;\n"

arquivo_sql = Path("/content/atividade4_sistema_academico.sql")
arquivo_sql.write_text(sql, encoding="utf-8")

print("Arquivo SQL criado em:", arquivo_sql)

# 4. Executa o SQL
with arquivo_sql.open("r", encoding="utf-8") as arquivo:
    execucao = subprocess.run(
        ["mariadb", "-u", "root", "--default-character-set=utf8mb4"],
        stdin=arquivo,
        text=True,
        capture_output=True
    )

if execucao.returncode != 0:
    print("ERRO DO MYSQL/MARIADB:")
    print(execucao.stderr)
    raise RuntimeError("Erro ao executar a Atividade 4.")

print("Banco sistema_academico criado com sucesso.")

# 5. Mostra os resultados
consultas = "USE sistema_academico;\n\nSELECT\n    'MÉDIAS DOS ALUNOS' AS etapa;\n\nSELECT\n    a.id_aluno,\n    a.nome,\n    ROUND(AVG(n.valor), 2) AS media\nFROM aluno AS a\nINNER JOIN nota AS n\n    ON n.id_aluno = a.id_aluno\nGROUP BY\n    a.id_aluno,\n    a.nome\nORDER BY a.id_aluno;\n\nSELECT\n    'MATRÍCULAS APÓS ATUALIZAÇÃO E EXCLUSÃO' AS etapa;\n\nSELECT\n    a.nome AS aluno,\n    c.nome AS curso,\n    m.data_matricula,\n    m.status\nFROM matricula AS m\nINNER JOIN aluno AS a\n    ON a.id_aluno = m.id_aluno\nINNER JOIN curso AS c\n    ON c.id_curso = m.id_curso\nORDER BY\n    a.nome,\n    c.nome;\n\nSELECT\n    'CONTAGEM FINAL' AS etapa;\n\nSELECT\n    'Alunos' AS tabela,\n    COUNT(*) AS quantidade\nFROM aluno\nUNION ALL\nSELECT 'Cursos', COUNT(*) FROM curso\nUNION ALL\nSELECT 'Matrículas', COUNT(*) FROM matricula\nUNION ALL\nSELECT 'Notas', COUNT(*) FROM nota;\n"

resultado = subprocess.run(
    [
        "mariadb",
        "-u",
        "root",
        "--table",
        "--default-character-set=utf8mb4",
        "-e",
        consultas
    ],
    text=True,
    capture_output=True
)

print(resultado.stdout)

if resultado.returncode != 0:
    print("ERRO DO MYSQL/MARIADB:")
    print(resultado.stderr)
    raise RuntimeError("Erro ao consultar os resultados.")

print("Atividade 4 executada com sucesso.")


Instalando MariaDB...
MariaDB iniciado com sucesso.
Arquivo SQL criado em: /content/atividade4_sistema_academico.sql
Banco sistema_academico criado com sucesso.
+--------------------+
| etapa              |
+--------------------+
| MÉDIAS DOS ALUNOS  |
+--------------------+
+----------+---------------------+-------+
| id_aluno | nome                | media |
+----------+---------------------+-------+
|        1 | Ana Beatriz Souza   |  8.50 |
|        2 | Bruno Henrique Lima |  5.50 |
|        3 | Carla Mendes Rocha  |  7.50 |
+----------+---------------------+-------+
+---------------------------------------------+
| etapa                                       |
+---------------------------------------------+
| MATRÍCULAS APÓS ATUALIZAÇÃO E EXCLUSÃO      |
+---------------------------------------------+
+---------------------+----------------------------+----------------+-----------+
| aluno               | curso                      | data_matricula | status    |
+------------------